In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import explained_variance_score, median_absolute_error, max_error
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 20, 'axes.titlesize': 20, 'axes.labelsize': 20,
    'xtick.labelsize': 20, 'ytick.labelsize': 20, 'legend.fontsize': 20,
    'figure.dpi': 300, 'savefig.dpi': 300,
})

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

Device: cuda


In [ ]:
PREPROCESSED_DIR = '/content/drive/MyDrive/DeepBudgetVis_Synthetic/preprocessed_10yr'
OUTPUT_DIR = '/content/drive/MyDrive/DeepBudgetVis_Synthetic/proposed_model_outputs_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEQ_LEN = 30          # lookback window, same as baselines - for a fair comparison
LAG_WINDOW = 21        # span the lag-attention module operates over, built around the
                        # ~14-day revenue-to-cash collection lag structurally present in the data
BATCH_SIZE = 16
MAX_EPOCHS = 120
PATIENCE = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4

In [ ]:
train_df = pd.read_csv(os.path.join(PREPROCESSED_DIR, 'train.csv'), parse_dates=['DATE'])
val_df   = pd.read_csv(os.path.join(PREPROCESSED_DIR, 'val_with_context.csv'), parse_dates=['DATE'])
test_df  = pd.read_csv(os.path.join(PREPROCESSED_DIR, 'test_with_context.csv'), parse_dates=['DATE'])
target_scaler = joblib.load(os.path.join(PREPROCESSED_DIR, 'target_scaler.joblib'))

with open(os.path.join(PREPROCESSED_DIR, 'preprocessing_metadata.json')) as f:
    meta = json.load(f)
target_cols = meta['target_cols']
target_cols_scaled = [f'{c}_scaled' for c in target_cols]

# Stream assignment - verified to exactly match the 39 pruned feature columns in your
# preprocessing metadata, nothing invented, nothing left out.
DEMAND_COLS = ['OCCUPANCY_RATE','STAFFED_BEDS','ADMISSIONS','ER_VISITS','OP_VISITS','SURGERIES',
               'DISCHARGES','AVG_LENGTH_OF_STAY','YEAR','IS_WEEKEND','QUARTER','IS_HOLIDAY',
               'DOW_SIN','DOW_COS','MONTH_SIN','MONTH_COS']
REVCYCLE_COLS = ['GROSS_CHARGES_MEDICARE','GROSS_CHARGES_MEDICAID','GROSS_CHARGES_COMMERCIAL',
                  'GROSS_CHARGES_SELFPAY','GROSS_CHARGES_OTHER','TOTAL_GROSS_CHARGES','CHARITY_CARE',
                  'BAD_DEBT','CLAIMS_SUBMITTED','DENIAL_RATE','CLAIMS_DENIED','CASH_COLLECTED']
EXPENSE_COLS = ['LABOR_EXP','SUPPLY_EXP','OVERHEAD_EXP','CAPITAL_EXP','OTHER_OPERATING_EXP',
                 'BUDGETED_LABOR_EXP','BUDGETED_SUPPLY_EXP','BUDGETED_PHARMACY_EXP',
                 'BUDGETED_OVERHEAD_EXP','BUDGET_VARIANCE','OPERATING_MARGIN_PCT']

# Verify this matches your ACTUAL preprocessing output before proceeding - if this assertion
# fails, your pruned feature set differs from the one this notebook was built against, and the
# stream lists below must be adjusted to match your real preprocessing_metadata.json.
assert set(DEMAND_COLS + REVCYCLE_COLS + EXPENSE_COLS) == set(meta['feature_cols']), \
    "Stream column lists do not match your actual pruned feature set - check preprocessing_metadata.json"

DEMAND_COLS_S = [f'{c}_scaled' for c in DEMAND_COLS]
REVCYCLE_COLS_S = [f'{c}_scaled' for c in REVCYCLE_COLS]
EXPENSE_COLS_S = [f'{c}_scaled' for c in EXPENSE_COLS]

def inverse_targets(y_scaled):
    return np.expm1(target_scaler.inverse_transform(y_scaled))

print(f"Demand stream: {len(DEMAND_COLS)} cols | Revenue-cycle stream: {len(REVCYCLE_COLS)} cols | "
      f"Expense stream: {len(EXPENSE_COLS)} cols")

Demand stream: 16 cols | Revenue-cycle stream: 12 cols | Expense stream: 11 cols


In [ ]:
class MultiStreamDataset(Dataset):
    """Unlike the baselines' single flat feature array, this returns THREE separate tensors
    per window - one per stream - so each encoder only ever sees its own mechanistically
    relevant columns."""
    def __init__(self, df, seq_len):
        self.Xd = df[DEMAND_COLS_S].values.astype(np.float32)
        self.Xr = df[REVCYCLE_COLS_S].values.astype(np.float32)
        self.Xe = df[EXPENSE_COLS_S].values.astype(np.float32)
        self.y = df[target_cols_scaled].values.astype(np.float32)
        self.is_anomaly = df['IS_ANOMALY'].values.astype(np.float32)  # AUXILIARY LABEL ONLY
        self.seq_len = seq_len
    def __len__(self):
        return len(self.Xd) - self.seq_len
    def __getitem__(self, idx):
        s = slice(idx, idx + self.seq_len)
        return (self.Xd[s], self.Xr[s], self.Xe[s], self.y[idx + self.seq_len],
                self.is_anomaly[idx + self.seq_len])

train_ds = MultiStreamDataset(train_df, SEQ_LEN)
val_ds = MultiStreamDataset(val_df, SEQ_LEN)
test_ds = MultiStreamDataset(test_df, SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

n_targets = len(target_cols)
print(f"Windows -> train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

Windows -> train: 3317, val: 153, test: 153


In [ ]:
class DemandEncoder(nn.Module):
    """1D-CNN (weekly local pattern, kernel=7) + 2-layer GRU (trend). Matches the architecture
    family already validated in the CNN-LSTM baseline, applied here only to demand/census columns."""
    def __init__(self, n_features, hidden=48, dropout=0.25):
        super().__init__()
        self.conv = nn.Conv1d(n_features, 32, kernel_size=7, padding=3)
        self.act = nn.ReLU()
        self.gru = nn.GRU(32, hidden, batch_first=True, dropout=dropout, num_layers=2)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x.transpose(1, 2); x = self.act(self.conv(x)); x = x.transpose(1, 2)
        out, _ = self.gru(x)
        return self.drop(out)


class RevCycleEncoder(nn.Module):
    """GRU + an explicit LAG-ATTENTION module over the last LAG_WINDOW days. This is Novelty 1's
    core mechanism: self-attention restricted to a window sized around the actual ~14-day
    revenue-to-cash collection lag your generator built into CASH_COLLECTED. No baseline model
    has this - it's the single biggest architectural difference from the LSTM/CNN-LSTM/Transformer
    baselines, which treat every day as equally relevant to every other day."""
    def __init__(self, n_features, hidden=48, lag_window=21, dropout=0.25):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True, dropout=dropout, num_layers=2)
        self.lag_window = lag_window
        self.lag_attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=4, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.gru(x)
        w = min(self.lag_window, out.size(1))
        recent = out[:, -w:, :]
        attn_out, _ = self.lag_attn(recent, recent, recent)
        fused = out.clone()
        fused[:, -w:, :] = fused[:, -w:, :] + attn_out
        return self.drop(fused)


class ExpenseEncoder(nn.Module):
    """Plain 2-layer GRU - no attention needed here, expense columns don't have the same
    explicit lag structure as the revenue-cycle stream."""
    def __init__(self, n_features, hidden=48, dropout=0.25):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True, dropout=dropout, num_layers=2)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.drop(out)

In [ ]:
class DeepBudgetVisForecaster(nn.Module):
    def __init__(self, n_demand, n_rev, n_exp, n_targets, hidden=48, dropout=0.25):
        super().__init__()
        self.demand_enc = DemandEncoder(n_demand, hidden, dropout)
        self.rev_enc = RevCycleEncoder(n_rev, hidden, LAG_WINDOW, dropout)
        self.exp_enc = ExpenseEncoder(n_exp, hidden, dropout)

        # Cross-stream fusion: treats the 3 streams' final hidden states as a 3-token sequence
        # and lets them attend to each other - e.g. lets the model learn "high ER_VISITS today
        # should raise attention on DENIAL_RATE" type cross-stream relationships.
        self.fusion_attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=4, dropout=dropout, batch_first=True)
        self.fusion_norm = nn.LayerNorm(hidden)

        # Novelty 3 - anomaly-conditioning: a small auxiliary head trained to predict IS_ANOMALY
        # FROM the fused representation (auxiliary loss, weight 0.1, added in the training loop).
        # IS_ANOMALY itself is NEVER an input - only a training signal for this head.
        self.anomaly_head = nn.Linear(hidden, 1)
        # The learned gate gets its signal from the SAME fused representation the anomaly head
        # reads - so as the anomaly head learns to recognize anomalous patterns, the gate
        # (trained jointly, not explicitly supervised) learns to down-weight them.
        self.gate = nn.Sequential(nn.Linear(hidden, hidden), nn.Sigmoid())

        self.forecast_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, n_targets)
        )

    def forward(self, xd, xr, xe):
        hd = self.demand_enc(xd)
        hr = self.rev_enc(xr)
        he = self.exp_enc(xe)

        stacked = torch.stack([hd[:, -1, :], hr[:, -1, :], he[:, -1, :]], dim=1)  # (batch, 3, hidden)
        fused, _ = self.fusion_attn(stacked, stacked, stacked)
        fused = self.fusion_norm(fused + stacked)          # residual connection, standard practice
        fused_pooled = fused.mean(dim=1)                    # (batch, hidden)

        anomaly_logit = self.anomaly_head(fused_pooled).squeeze(-1)
        gate_weights = self.gate(fused_pooled)
        gated = fused_pooled * gate_weights

        forecast = self.forecast_head(gated)
        return forecast, anomaly_logit


model = DeepBudgetVisForecaster(len(DEMAND_COLS), len(REVCYCLE_COLS), len(EXPENSE_COLS), n_targets).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params:,}")

Total trainable parameters: 99,235


In [ ]:
def compute_metrics(y_true, y_pred, y_true_scaled=None, y_pred_scaled=None):
    """10 metrics, chosen to cover: typical error (MAE), large-error sensitivity (RMSE),
    relative/business-communicable error (MAPE, SMAPE), model-fit quality (R2, ExplainedVar),
    robustness to outliers (MedianAE), worst-case risk (MaxError - relevant for budget shortfalls),
    and error on the SAME scale the model actually trains on (MAE_scaled, RMSE_scaled - the
    correction you asked for a few turns ago)."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denom = np.where(np.abs(y_true) < 1e-6, 1e-6, np.abs(y_true))
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    smape = np.mean(2*np.abs(y_true-y_pred) / (np.abs(y_true)+np.abs(y_pred)+1e-6)) * 100
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 1e-12 else float('nan')
    evs = explained_variance_score(y_true.flatten(), y_pred.flatten())
    medae = median_absolute_error(y_true.flatten(), y_pred.flatten())
    maxerr = max_error(y_true.flatten(), y_pred.flatten())
    mae_s = np.mean(np.abs(y_true_scaled - y_pred_scaled)) if y_true_scaled is not None else float('nan')
    rmse_s = np.sqrt(np.mean((y_true_scaled - y_pred_scaled)**2)) if y_true_scaled is not None else float('nan')
    return {'MAE': float(mae), 'RMSE': float(rmse), 'MAPE': float(mape), 'SMAPE': float(smape),
            'R2': float(r2), 'ExplainedVar': float(evs), 'MedianAE': float(medae),
            'MaxError': float(maxerr), 'MAE_scaled': float(mae_s), 'RMSE_scaled': float(rmse_s)}

In [ ]:
criterion_forecast = nn.MSELoss()
criterion_anomaly = nn.BCEWithLogitsLoss()   # for the auxiliary anomaly-prediction head

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# LR scheduler 1: linear warmup for the first 3 epochs. Attention layers (used in the
# lag-attention and fusion modules) are known to train more stably with a short warmup rather
# than the full learning rate from step 1 - this is standard practice for attention-containing
# architectures, not used in the baselines since they use plain LSTM/CNN with less attention.
warmup_epochs = 3
def warmup_lr(epoch):
    return min(1.0, (epoch + 1) / warmup_epochs)
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lr)

# LR scheduler 2: ReduceLROnPlateau, same as the baselines - takes over AFTER warmup completes,
# halving LR when val loss plateaus for 4 epochs.
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING
# Add this cell AFTER Cell 7 and BEFORE Cell 8.
#
# Everything else in the notebook remains unchanged.
# ============================================================

import itertools
import copy
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Search space
# These are ONLY existing model/training hyperparameters.
# ------------------------------------------------------------
SEARCH_SPACE = {
    "hidden": [32, 48, 64],
    "dropout": [0.20, 0.30, 0.40],
    "lr": [3e-4, 5e-4, 1e-3, 2e-3],
    "weight_decay": [1e-5, 1e-4, 5e-4],
    "lag_window": [14, 21, 28],
    "batch_size": [16, 32],
}

# Number of randomly selected configurations.
# This avoids exhaustively training all combinations.
N_TRIALS = 12

# Shorter training during tuning only.
# FINAL training below will still use the notebook's
# original MAX_EPOCHS=60 and PATIENCE=10.
TUNE_MAX_EPOCHS = 30
TUNE_PATIENCE = 5


def set_trial_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_trial_loaders(batch_size):
    train_loader_trial = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader_trial = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader_trial, val_loader_trial


def run_tuning_trial(params, trial_number):
    """
    Train ONE hyperparameter configuration using the
    existing architecture and the existing train/validation data.

    Selection criterion:
        validation combined loss =
        forecast MSE + 0.1 * anomaly BCE

    The test set is NEVER used here.
    """

    set_trial_seed(SEED)

    # Build model using the SAME architecture as Cell 5.
    trial_model = DeepBudgetVisForecaster(
        len(DEMAND_COLS),
        len(REVCYCLE_COLS),
        len(EXPENSE_COLS),
        n_targets,
        hidden=params["hidden"],
        dropout=params["dropout"]
    ).to(device)

    # Replace only the lag-window hyperparameter inside the
    # existing RevCycleEncoder instances.
    trial_model.rev_enc.lag_window = params["lag_window"]

    train_loader_trial, val_loader_trial = make_trial_loaders(
        params["batch_size"]
    )

    criterion_forecast_trial = nn.MSELoss()
    criterion_anomaly_trial = nn.BCEWithLogitsLoss()

    optimizer_trial = torch.optim.Adam(
        trial_model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )

    # Same 3-epoch warm-up as the original notebook.
    warmup_epochs_trial = 3

    def warmup_lr_trial(epoch):
        return min(1.0, (epoch + 1) / warmup_epochs_trial)

    warmup_scheduler_trial = torch.optim.lr_scheduler.LambdaLR(
        optimizer_trial,
        lr_lambda=warmup_lr_trial
    )

    # Same plateau scheduler as the original notebook.
    plateau_scheduler_trial = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_trial,
        mode='min',
        factor=0.5,
        patience=4
    )

    best_val_loss = float("inf")
    best_epoch = -1
    epochs_without_improvement = 0

    for epoch in range(TUNE_MAX_EPOCHS):

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------
        trial_model.train()
        train_losses = []

        for xd, xr, xe, y, is_anom in train_loader_trial:

            xd = xd.to(device)
            xr = xr.to(device)
            xe = xe.to(device)
            y = y.to(device)
            is_anom = is_anom.to(device)

            optimizer_trial.zero_grad()

            pred, anom_logit = trial_model(xd, xr, xe)

            loss = (
                criterion_forecast_trial(pred, y)
                + 0.1 * criterion_anomaly_trial(anom_logit, is_anom)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trial_model.parameters(),
                max_norm=1.0
            )

            optimizer_trial.step()

            train_losses.append(loss.item())

        if epoch < warmup_epochs_trial:
            warmup_scheduler_trial.step()

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------
        trial_model.eval()
        val_losses = []

        with torch.no_grad():

            for xd, xr, xe, y, is_anom in val_loader_trial:

                xd = xd.to(device)
                xr = xr.to(device)
                xe = xe.to(device)
                y = y.to(device)
                is_anom = is_anom.to(device)

                pred, anom_logit = trial_model(xd, xr, xe)

                val_loss = (
                    criterion_forecast_trial(pred, y)
                    + 0.1 * criterion_anomaly_trial(
                        anom_logit,
                        is_anom
                    )
                )

                val_losses.append(val_loss.item())

        mean_val_loss = float(np.mean(val_losses))

        if epoch >= warmup_epochs_trial:
            plateau_scheduler_trial.step(mean_val_loss)

        # ----------------------------------------------------
        # Early stopping for THIS tuning trial
        # ----------------------------------------------------
        if mean_val_loss < best_val_loss - 1e-6:

            best_val_loss = mean_val_loss
            best_epoch = epoch
            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= TUNE_PATIENCE:
            break

    result = {
        "trial": trial_number,
        "hidden": params["hidden"],
        "dropout": params["dropout"],
        "lr": params["lr"],
        "weight_decay": params["weight_decay"],
        "lag_window": params["lag_window"],
        "batch_size": params["batch_size"],
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch + 1,
    }

    # Free trial model memory before next trial.
    del trial_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ------------------------------------------------------------
# Generate reproducible candidate configurations
# ------------------------------------------------------------
all_configs = list(
    itertools.product(
        SEARCH_SPACE["hidden"],
        SEARCH_SPACE["dropout"],
        SEARCH_SPACE["lr"],
        SEARCH_SPACE["weight_decay"],
        SEARCH_SPACE["lag_window"],
        SEARCH_SPACE["batch_size"],
    )
)

rng = random.Random(SEED)
rng.shuffle(all_configs)

selected_configs = all_configs[:N_TRIALS]

print("=" * 80)
print("DEEPBUDGET-VIS HYPERPARAMETER TUNING")
print("=" * 80)
print(f"Total possible combinations: {len(all_configs)}")
print(f"Trials to run: {len(selected_configs)}")
print(f"Device: {device}")
print()

# ------------------------------------------------------------
# Run tuning
# ------------------------------------------------------------
tuning_results = []

for trial_number, values in enumerate(selected_configs, start=1):

    params = {
        "hidden": values[0],
        "dropout": values[1],
        "lr": values[2],
        "weight_decay": values[3],
        "lag_window": values[4],
        "batch_size": values[5],
    }

    print(
        f"Trial {trial_number:02d}/{len(selected_configs)} | "
        f"hidden={params['hidden']} | "
        f"dropout={params['dropout']} | "
        f"lr={params['lr']} | "
        f"wd={params['weight_decay']} | "
        f"lag={params['lag_window']} | "
        f"batch={params['batch_size']}"
    )

    result = run_tuning_trial(
        params,
        trial_number
    )

    tuning_results.append(result)

    print(
        f"   best_val_loss={result['best_val_loss']:.6f} | "
        f"best_epoch={result['best_epoch']}"
    )

# ------------------------------------------------------------
# Rank configurations
# ------------------------------------------------------------
tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values(
    "best_val_loss",
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 80)
print("TOP HYPERPARAMETER CONFIGURATIONS")
print("=" * 80)

print(
    tuning_df.head(10).to_string(index=False)
)

# ------------------------------------------------------------
# Select best configuration
# ------------------------------------------------------------
best_params = {
    "hidden": int(tuning_df.loc[0, "hidden"]),
    "dropout": float(tuning_df.loc[0, "dropout"]),
    "lr": float(tuning_df.loc[0, "lr"]),
    "weight_decay": float(tuning_df.loc[0, "weight_decay"]),
    "lag_window": int(tuning_df.loc[0, "lag_window"]),
    "batch_size": int(tuning_df.loc[0, "batch_size"]),
}

print("\n" + "=" * 80)
print("BEST CONFIGURATION")
print("=" * 80)

for k, v in best_params.items():
    print(f"{k}: {v}")

# ------------------------------------------------------------
# IMPORTANT:
# Now configure the EXISTING notebook globals for FINAL training.
# The architecture itself is unchanged.
# ------------------------------------------------------------

HIDDEN = best_params["hidden"]
DROPOUT = best_params["dropout"]
LR = best_params["lr"]
WEIGHT_DECAY = best_params["weight_decay"]
LAG_WINDOW = best_params["lag_window"]
BATCH_SIZE = best_params["batch_size"]

# ------------------------------------------------------------
# Rebuild the final model using the BEST configuration.
# ------------------------------------------------------------
set_trial_seed(SEED)

model = DeepBudgetVisForecaster(
    len(DEMAND_COLS),
    len(REVCYCLE_COLS),
    len(EXPENSE_COLS),
    n_targets,
    hidden=HIDDEN,
    dropout=DROPOUT
).to(device)

# ------------------------------------------------------------
# Rebuild the FINAL optimizer/schedulers using best params.
# These otherwise remain exactly as in the original notebook.
# ------------------------------------------------------------
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

warmup_epochs = 3

def warmup_lr(epoch):
    return min(1.0, (epoch + 1) / warmup_epochs)

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=warmup_lr
)

plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=4
)

# ------------------------------------------------------------
# Rebuild loaders using the BEST batch size.
# Train/validation/test DATA itself is unchanged.
# ------------------------------------------------------------
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ------------------------------------------------------------
# Use a NEW output directory so the old checkpoint cannot
# accidentally be resumed by Cell 8.
# ------------------------------------------------------------
TUNED_OUTPUT_DIR = os.path.join(
    os.path.dirname(OUTPUT_DIR),
    "proposed_model_outputs_v1_tuned"
)

os.makedirs(TUNED_OUTPUT_DIR, exist_ok=True)
OUTPUT_DIR = TUNED_OUTPUT_DIR

# Save tuning results and selected parameters.
tuning_df.to_csv(
    os.path.join(OUTPUT_DIR, "hyperparameter_results.csv"),
    index=False
)

with open(
    os.path.join(OUTPUT_DIR, "best_hyperparameters.json"),
    "w"
) as f:
    json.dump(best_params, f, indent=2)

print("\n" + "=" * 80)
print("TUNING COMPLETE")
print("=" * 80)
print(f"Final training will use:")
print(f"  hidden       = {HIDDEN}")
print(f"  dropout     = {DROPOUT}")
print(f"  learning_rate = {LR}")
print(f"  weight_decay = {WEIGHT_DECAY}")
print(f"  lag_window   = {LAG_WINDOW}")
print(f"  batch_size   = {BATCH_SIZE}")
print(f"\nFinal output directory:")
print(f"  {OUTPUT_DIR}")
print("\nNow run Cell 8 and continue normally.")

DEEPBUDGET-VIS HYPERPARAMETER TUNING
Total possible combinations: 648
Trials to run: 12
Device: cuda

Trial 01/12 | hidden=32 | dropout=0.3 | lr=0.002 | wd=0.0001 | lag=14 | batch=16
   best_val_loss=0.167771 | best_epoch=6
Trial 02/12 | hidden=64 | dropout=0.2 | lr=0.001 | wd=0.0005 | lag=28 | batch=32
   best_val_loss=0.163398 | best_epoch=10
Trial 03/12 | hidden=48 | dropout=0.2 | lr=0.0005 | wd=0.0001 | lag=28 | batch=32
   best_val_loss=0.161663 | best_epoch=8
Trial 04/12 | hidden=64 | dropout=0.2 | lr=0.0003 | wd=0.0005 | lag=28 | batch=16
   best_val_loss=0.165728 | best_epoch=4
Trial 05/12 | hidden=48 | dropout=0.2 | lr=0.0005 | wd=0.0001 | lag=14 | batch=16
   best_val_loss=0.162134 | best_epoch=8
Trial 06/12 | hidden=32 | dropout=0.3 | lr=0.001 | wd=1e-05 | lag=28 | batch=32
   best_val_loss=0.161207 | best_epoch=10
Trial 07/12 | hidden=32 | dropout=0.4 | lr=0.001 | wd=0.0005 | lag=28 | batch=32
   best_val_loss=0.169379 | best_epoch=10
Trial 08/12 | hidden=32 | dropout=0.4 |

In [ ]:
def train_model(max_epochs=MAX_EPOCHS, patience=PATIENCE):
    ckpt_path = os.path.join(OUTPUT_DIR, 'checkpoint.pt')
    log_path = os.path.join(OUTPUT_DIR, 'history.csv')

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0
    history = []

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        plateau_scheduler.load_state_dict(ckpt['plateau_scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt['best_val_loss']
        epochs_no_improve = ckpt['epochs_no_improve']
        history = ckpt['history']
        print(f"Resuming from epoch {start_epoch} (best_val_loss so far: {best_val_loss:.4f})")
    else:
        print("No checkpoint found - starting fresh from epoch 0")

    for epoch in range(start_epoch, max_epochs):
        model.train()
        train_losses, train_true, train_pred = [], [], []
        for xd, xr, xe, y, is_anom in train_loader:
            xd, xr, xe, y, is_anom = xd.to(device), xr.to(device), xe.to(device), y.to(device), is_anom.to(device)
            optimizer.zero_grad()
            pred, anom_logit = model(xd, xr, xe)
            loss = criterion_forecast(pred, y) + 0.1 * criterion_anomaly(anom_logit, is_anom)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())
            train_true.append(y.detach().cpu().numpy())
            train_pred.append(pred.detach().cpu().numpy())
        if epoch < warmup_epochs:
            warmup_scheduler.step()

        model.eval()
        val_losses, val_true, val_pred = [], [], []
        with torch.no_grad():
            for xd, xr, xe, y, is_anom in val_loader:
                xd, xr, xe, y, is_anom = xd.to(device), xr.to(device), xe.to(device), y.to(device), is_anom.to(device)
                pred, anom_logit = model(xd, xr, xe)
                loss = criterion_forecast(pred, y) + 0.1 * criterion_anomaly(anom_logit, is_anom)
                val_losses.append(loss.item())
                val_true.append(y.cpu().numpy())
                val_pred.append(pred.cpu().numpy())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        if epoch >= warmup_epochs:
            plateau_scheduler.step(val_loss)

        train_true_s = np.concatenate(train_true); train_pred_s = np.concatenate(train_pred)
        val_true_s = np.concatenate(val_true); val_pred_s = np.concatenate(val_pred)
        train_metrics = compute_metrics(inverse_targets(train_true_s), inverse_targets(train_pred_s), train_true_s, train_pred_s)
        val_metrics = compute_metrics(inverse_targets(val_true_s), inverse_targets(val_pred_s), val_true_s, val_pred_s)

        row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
               'lr': optimizer.param_groups[0]['lr']}
        row.update({f'train_{k}': v for k, v in train_metrics.items()})
        row.update({f'val_{k}': v for k, v in val_metrics.items()})
        history.append(row)

        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
              f"val_MAE={val_metrics['MAE']:.0f} val_RMSE={val_metrics['RMSE']:.0f} "
              f"val_MAPE={val_metrics['MAPE']:.2f}% val_R2={val_metrics['R2']:.3f} "
              f"val_SMAPE={val_metrics['SMAPE']:.2f}% val_MedianAE={val_metrics['MedianAE']:.0f} "
              f"val_MaxError={val_metrics['MaxError']:.0f} val_ExplainedVar={val_metrics['ExplainedVar']:.3f} "
              f"val_MAE_scaled={val_metrics['MAE_scaled']:.4f} val_RMSE_scaled={val_metrics['RMSE_scaled']:.4f}")

        improved = val_loss < best_val_loss - 1e-6
        if improved:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pt'))
        else:
            epochs_no_improve += 1

        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'plateau_scheduler_state': plateau_scheduler.state_dict(),
            'best_val_loss': best_val_loss, 'epochs_no_improve': epochs_no_improve, 'history': history,
        }, ckpt_path)
        pd.DataFrame(history).to_csv(log_path, index=False)

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}. Best val_loss={best_val_loss:.4f}")
            break

    return pd.DataFrame(history)

In [ ]:
history_df = train_model()
print("\nTraining complete (or resumed to completion).")

No checkpoint found - starting fresh from epoch 0
Epoch 000 | train_loss=0.6974 val_loss=0.5034 | val_MAE=12054 val_RMSE=16147 val_MAPE=6.97% val_R2=-0.689 val_SMAPE=7.27% val_MedianAE=6559 val_MaxError=49822 val_ExplainedVar=0.880 val_MAE_scaled=0.5680 val_RMSE_scaled=0.6858
Epoch 001 | train_loss=0.3170 val_loss=0.2683 | val_MAE=7386 val_RMSE=10839 val_MAPE=4.24% val_R2=0.239 val_SMAPE=4.40% val_MedianAE=4056 val_MaxError=47670 val_ExplainedVar=0.949 val_MAE_scaled=0.3449 val_RMSE_scaled=0.4992
Epoch 002 | train_loss=0.2262 val_loss=0.1997 | val_MAE=5161 val_RMSE=8337 val_MAPE=3.03% val_R2=0.550 val_SMAPE=3.11% val_MedianAE=2347 val_MaxError=44849 val_ExplainedVar=0.961 val_MAE_scaled=0.2469 val_RMSE_scaled=0.4253
Epoch 003 | train_loss=0.1931 val_loss=0.1810 | val_MAE=4799 val_RMSE=7728 val_MAPE=2.89% val_R2=0.613 val_SMAPE=2.92% val_MedianAE=2569 val_MaxError=43681 val_ExplainedVar=0.961 val_MAE_scaled=0.2374 val_RMSE_scaled=0.4049
Epoch 004 | train_loss=0.1870 val_loss=0.2051 | va

In [ ]:
metric_names = ['MAE', 'RMSE', 'MAPE', 'SMAPE', 'R2', 'ExplainedVar', 'MedianAE',
                 'MaxError', 'MAE_scaled', 'RMSE_scaled']

fig, axes = plt.subplots(4, 3, figsize=(30, 32))
axes_flat = axes.flatten()
for i, m in enumerate(metric_names):
    ax = axes_flat[i]
    ax.plot(history_df['epoch'], history_df[f'train_{m}'], label='Train', linewidth=2.5)
    ax.plot(history_df['epoch'], history_df[f'val_{m}'], label='Val', linewidth=2.5)
    ax.set_title(m, fontsize=20)
    ax.set_xlabel('Epoch', fontsize=20)
    ax.legend(fontsize=18)
axes_flat[10].plot(history_df['epoch'], history_df['train_loss'], label='Train', linewidth=2.5)
axes_flat[10].plot(history_df['epoch'], history_df['val_loss'], label='Val', linewidth=2.5)
axes_flat[10].set_title('Loss (forecast MSE + 0.1*anomaly BCE)', fontsize=20)
axes_flat[10].legend(fontsize=18)
axes_flat[11].plot(history_df['epoch'], history_df['lr'], color='green', linewidth=2.5)
axes_flat[11].set_title('Learning Rate', fontsize=20)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

Saved: /content/drive/MyDrive/DeepBudgetVis_Synthetic/proposed_model_outputs_v1_tuned/training_curves.png


In [ ]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pt'), map_location=device))
model.eval()

all_true, all_pred, all_anom_flags = [], [], []
with torch.no_grad():
    for xd, xr, xe, y, is_anom in test_loader:
        xd, xr, xe = xd.to(device), xr.to(device), xe.to(device)
        pred, _ = model(xd, xr, xe)
        all_true.append(y.numpy()); all_pred.append(pred.cpu().numpy())
        all_anom_flags.append(is_anom.numpy())

y_true_s = np.concatenate(all_true); y_pred_s = np.concatenate(all_pred)
anom_flags = np.concatenate(all_anom_flags)
y_true = inverse_targets(y_true_s); y_pred = inverse_targets(y_pred_s)

overall = compute_metrics(y_true, y_pred, y_true_s, y_pred_s)
anom_mask = anom_flags == 1
normal_mask = ~anom_mask
anom_metrics = compute_metrics(y_true[anom_mask], y_pred[anom_mask],
                                 y_true_s[anom_mask], y_pred_s[anom_mask]) if anom_mask.sum() > 0 else None
normal_metrics = compute_metrics(y_true[normal_mask], y_pred[normal_mask],
                                   y_true_s[normal_mask], y_pred_s[normal_mask]) if normal_mask.sum() > 0 else None

print("TEST metrics (overall):", overall)
print(f"Anomaly windows: {anom_mask.sum()}/{len(y_true)}")
if anom_metrics: print("  anomaly:", anom_metrics)
if normal_metrics: print("  normal: ", normal_metrics)

metrics_df = pd.DataFrame([
    {'split': 'test_overall', **overall},
    {'split': 'test_anomaly', **(anom_metrics or {})},
    {'split': 'test_normal', **(normal_metrics or {})},
])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'metrics.csv'), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_DIR, 'metrics.csv')}")

TEST metrics (overall): {'MAE': 4548.50244140625, 'RMSE': 7190.4716796875, 'MAPE': 2.820634365081787, 'SMAPE': 2.789571523666382, 'R2': 0.6836812496185303, 'ExplainedVar': 0.9651216864585876, 'MedianAE': 2258.79296875, 'MaxError': 35894.65625, 'MAE_scaled': 0.22123928368091583, 'RMSE_scaled': 0.34430748224258423}
Anomaly windows: 3/153
  anomaly: {'MAE': 13848.2177734375, 'RMSE': 16140.5, 'MAPE': 8.849787712097168, 'SMAPE': 9.379705429077148, 'R2': -0.8137108087539673, 'ExplainedVar': 0.8035837411880493, 'MedianAE': 12057.890625, 'MaxError': 29013.2265625, 'MAE_scaled': 0.9642112851142883, 'RMSE_scaled': 1.3260440826416016}
  normal:  {'MAE': 4362.50830078125, 'RMSE': 6893.95556640625, 'MAPE': 2.7000513076782227, 'SMAPE': 2.657768726348877, 'R2': 0.7034039497375488, 'ExplainedVar': 0.9684442281723022, 'MedianAE': 2225.21484375, 'MaxError': 35894.65625, 'MAE_scaled': 0.20637983083724976, 'RMSE_scaled': 0.29283225536346436}

Saved: /content/drive/MyDrive/DeepBudgetVis_Synthetic/proposed_

In [ ]:
print("="*70)
print("DEEPBUDGET-VIS PROPOSED MODEL (Novelty 1 + 3) - FINAL SUMMARY")
print("="*70)
print(f"Architecture: 3-stream encoder (Demand: CNN+GRU | RevCycle: GRU+LagAttention "
      f"| Expense: GRU) -> cross-stream fusion attention -> anomaly-conditioned gate -> forecast head")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trained epochs: {len(history_df)} (early-stopped: {len(history_df) < MAX_EPOCHS})")
print(f"\nTest set results (combined revenue+expense):")
for k, v in overall.items():
    print(f"  {k}: {v:.4f}")
print(f"\nAnomaly-day degradation check (R2, normal vs anomaly windows):")
if normal_metrics and anom_metrics:
    print(f"  Normal windows R2: {normal_metrics['R2']:.3f} | Anomaly windows R2: {anom_metrics['R2']:.3f}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("  - checkpoint.pt (resume-capable)")
print("  - best_model.pt (lowest val_loss checkpoint)")
print("  - history.csv (all 10 metrics, every epoch, train+val)")
print("  - metrics.csv (final test metrics: overall/anomaly/normal)")
print("  - training_curves.png (all 10 metrics + loss + LR, fontsize=20/dpi=300)")
print("\nNOTE: this model does NOT yet include Novelty 2 (uncertainty-gated budget")
print("optimization) - deferred per the incremental-testing plan. Compare the R2/MAE/RMSE")
print("above directly against the baseline notebook's test_summary.csv for the same metrics.")

DEEPBUDGET-VIS PROPOSED MODEL (Novelty 1 + 3) - FINAL SUMMARY
Architecture: 3-stream encoder (Demand: CNN+GRU | RevCycle: GRU+LagAttention | Expense: GRU) -> cross-stream fusion attention -> anomaly-conditioned gate -> forecast head
Parameters: 48,419
Trained epochs: 25 (early-stopped: True)

Test set results (combined revenue+expense):
  MAE: 4548.5024
  RMSE: 7190.4717
  MAPE: 2.8206
  SMAPE: 2.7896
  R2: 0.6837
  ExplainedVar: 0.9651
  MedianAE: 2258.7930
  MaxError: 35894.6562
  MAE_scaled: 0.2212
  RMSE_scaled: 0.3443

Anomaly-day degradation check (R2, normal vs anomaly windows):
  Normal windows R2: 0.703 | Anomaly windows R2: -0.814

All outputs saved to: /content/drive/MyDrive/DeepBudgetVis_Synthetic/proposed_model_outputs_v1_tuned
  - checkpoint.pt (resume-capable)
  - best_model.pt (lowest val_loss checkpoint)
  - history.csv (all 10 metrics, every epoch, train+val)
  - metrics.csv (final test metrics: overall/anomaly/normal)
  - training_curves.png (all 10 metrics + loss + 